In [1]:
!pip install matplotlib geopandas rasterio contextily mapclassify matplotlib-scalebar

In [2]:
import matplotlib.pyplot as plt   # motor de dibujo (figura/ejes/colores)
import geopandas as gpd             # vectoriales (extiende pandas con geometrías)
import rasterio                     # raster (GeoTIFF + metadatos georreferenciados)
from rasterio.plot import show      # helper para dibujar un raster en un eje matplotlib
import contextily as ctx            # tiles de basemap web (OSM, CartoDB, etc.)
import numpy as np                  # álgebra de matrices (rasters son matrices)
from rasterio.mask import mask
from matplotlib_scalebar.scalebar import ScaleBar
from rasterio.warp import calculate_default_transform, reproject, Resampling



# Carga de datos

In [7]:
centrales = gpd.read_file('Paneles.gdb')
codigos_norte = [2, 3, '2', '3', '02', '03']
Fotovoltaicas_filtradas = centrales[centrales['REGION'].isin(codigos_norte)]

len(Fotovoltaicas_filtradas)

105

In [5]:
almacenamiento = gpd.read_file('Almacenamiento/Almacenamiento_de_Energía.shp')

In [4]:
subestaciones = gpd.read_file('Subestaciones/Subestaciones.shp')

# Procesamiento

In [14]:
crs_proyectado = 32719

plantas_solares = centrales.to_crs(epsg=crs_proyectado)
subestaciones = subestaciones.to_crs(epsg=crs_proyectado)
almacenamiento = almacenamiento.to_crs(epsg=crs_proyectado)

plantas_con_subestacion = gpd.sjoin_nearest(
    plantas_solares,
    subestaciones,
    how='left',
    distance_col='dist_subestacion_m'
)

plantas_con_almacenamiento = gpd.sjoin_nearest(
    plantas_solares,
    almacenamiento,
    how='left',
    distance_col='dist_almacenamiento_m'
)

dist_sub_km = plantas_con_subestacion['dist_subestacion_m'] / 1000
dist_alm_km = plantas_con_almacenamiento['dist_almacenamiento_m'] / 1000


print("Datos Subestaciones")
print(f"Mínimo (Planta mejor ubicada):       {dist_sub_km.min():.2f} km")
print(f"Máximo (Planta peor ubicada):        {dist_sub_km.max():.2f} km")
print(f"Promedio (Media):                    {dist_sub_km.mean():.2f} km")
print(f"Mediana:                             {dist_sub_km.median():.2f} km")
print(f"Desviación Estándar:                 {dist_sub_km.std():.2f} km")
print("\n")
print("Datos almacen")
print(f"Mínimo (Planta mejor ubicada):       {dist_alm_km.min():.2f} km")
print(f"Máximo (Planta peor ubicada):        {dist_alm_km.max():.2f} km")
print(f"Promedio (Media):                    {dist_alm_km.mean():.2f} km")
print(f"Mediana:                             {dist_alm_km.median():.2f} km")
print(f"Desviación Estándar:                 {dist_alm_km.std():.2f} km")


Datos Subestaciones
Mínimo (Planta mejor ubicada):       0.04 km
Máximo (Planta peor ubicada):        32.36 km
Promedio (Media):                    4.39 km
Mediana:                             3.38 km
Desviación Estándar:                 3.99 km


Datos almacen
Mínimo (Planta mejor ubicada):       0.24 km
Máximo (Planta peor ubicada):        785.96 km
Promedio (Media):                    95.63 km
Mediana:                             79.45 km
Desviación Estándar:                 69.92 km
